# ResNet-50 at 320x320

## 1. Setup

In [ ]:
import gc
import os
import sys
from statistics import mean, stdev
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / 'src').exists():
    raise RuntimeError("Folder src not found. Let's open notebook from this repo.")

os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print('Project root:', PROJECT_ROOT)

## 2. Import shared code

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import transforms
from datasets import load_dataset

from src.datasets.glaucoma_dataset import GlaucomaDataset, IDX_TO_CLASS, _drop_duplicates
from src.models.resnet50 import ResNet50Classifier
from src.training.config import (
    CHECKPOINT_DIR, DEVICE, LEARNING_RATE, NUM_CLASSES, set_seed
)
from src.training.train_loop import evaluate
from src.utils.metrics import compute_metrics, save_metrics, print_report
from src.utils.visualize import plot_training_curves, plot_confusion_matrix

if not torch.cuda.is_available():
    raise RuntimeError('CUDA is not available in this notebook kernel')

set_seed()

print('PyTorch:', torch.__version__)
print('GPU:', torch.cuda.get_device_name(0))

## 3. Configuration

In [ ]:
IMAGE_SIZE = 320
BATCH_SIZE_CANDIDATES = (16, 8, 4)  # falls back to a smaller batch size on CUDA OOM
AMP_ENABLED = True
SEEDS = (42, 123, 2026)
WARMUP_EPOCHS = 5
FINETUNE_EPOCHS = 40
PATIENCE = 7
MIN_DELTA = 1e-4
WARMUP_LR = 1e-3
BACKBONE_LR = 1e-5
HEAD_LR = LEARNING_RATE
EXPERIMENT_WEIGHT_DECAY = 1e-3
LABEL_SMOOTHING = 0.1
MODEL_NAME = 'resnet50_warmup'
CLASS_NAMES = [IDX_TO_CLASS[i] for i in range(NUM_CLASSES)]


## 4. 320x320 data pipeline

In [ ]:
train_transform_320 = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=10),
    transforms.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.05),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225],
    ),
])

eval_transform_320 = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225],
    ),
])

raw_dataset = load_dataset('moondream/glaucoma-detection')
train_set = _drop_duplicates(raw_dataset['train'])
val_set = _drop_duplicates(raw_dataset['validation'])
test_set = _drop_duplicates(raw_dataset['test'])

def build_experiment_loaders(batch_size):
    train_data = GlaucomaDataset(train_set, transform=train_transform_320)
    val_data = GlaucomaDataset(val_set, transform=eval_transform_320)
    test_data = GlaucomaDataset(test_set, transform=eval_transform_320)
    return (
        DataLoader(train_data, batch_size=batch_size, shuffle=True),
        DataLoader(val_data, batch_size=batch_size, shuffle=False),
        DataLoader(test_data, batch_size=batch_size, shuffle=False),
    )

## 5. Two-stage training with AMP and an automatic OOM fallback

In [ ]:
def fit_resnet50(model, loss_fn, train_loader, val_loader, checkpoint_path):
    history = {'train_loss': [], 'val_loss': []}
    scaler = torch.amp.GradScaler('cuda', enabled=AMP_ENABLED)
    best_val_loss = float('inf')
    best_epoch = None
    global_epoch = 0

    def run_phase(name, epochs, optimizer, freeze_backbone, early_stopping):
        nonlocal best_val_loss, best_epoch, global_epoch
        stale = 0
        for phase_epoch in range(1, epochs + 1):
            model.train()
            if freeze_backbone:
                model.model.eval()  # keep frozen BatchNorm statistics unchanged
                model.model.fc.train()

            total_train_loss = 0.0
            for images, labels in train_loader:
                images, labels = images.to(DEVICE), labels.to(DEVICE)
                optimizer.zero_grad()
                with torch.autocast(device_type='cuda', dtype=torch.float16, enabled=AMP_ENABLED):
                    loss = loss_fn(model(images), labels)
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
                total_train_loss += loss.item() * images.size(0)

            global_epoch += 1
            train_loss = total_train_loss / len(train_loader.dataset)
            val_loss, val_preds, val_labels = evaluate(model, val_loader, loss_fn)
            val_metrics = compute_metrics(val_labels, val_preds)
            history['train_loss'].append(train_loss)
            history['val_loss'].append(val_loss)

            improved = val_loss < best_val_loss - MIN_DELTA
            print(
                f'{name} {phase_epoch}/{epochs} - train_loss: {train_loss:.4f} | '
                f"val_loss: {val_loss:.4f} - val_acc: {val_metrics['accuracy']:.4f} | "
                f"val_f1_macro: {val_metrics['f1_macro']:.4f}"
                + ('  (best validation loss)' if improved else '')
            )

            if improved:
                best_val_loss = val_loss
                best_epoch = global_epoch
                stale = 0
                checkpoint_path.parent.mkdir(parents=True, exist_ok=True)
                torch.save(model.state_dict(), checkpoint_path)
            else:
                stale += 1
                if early_stopping and stale >= PATIENCE:
                    print(f'No validation-loss improvement for {PATIENCE} epochs; stopping early.')
                    return

    warmup_optimizer = torch.optim.AdamW(
        model.model.fc.parameters(), lr=WARMUP_LR, weight_decay=EXPERIMENT_WEIGHT_DECAY
    )
    run_phase('Warm-up', WARMUP_EPOCHS, warmup_optimizer, True, False)

    model.unfreeze_features()
    backbone_parameters = [
        parameter for name, parameter in model.model.named_parameters()
        if not name.startswith('fc.')
    ]
    finetune_optimizer = torch.optim.AdamW([
        {'params': backbone_parameters, 'lr': BACKBONE_LR},
        {'params': model.model.fc.parameters(), 'lr': HEAD_LR},
    ], weight_decay=EXPERIMENT_WEIGHT_DECAY)
    run_phase('Fine-tune', FINETUNE_EPOCHS, finetune_optimizer, False, True)

    return history, best_epoch

In [ ]:
runs = []
loss_fn = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)

for run_seed in SEEDS:
    checkpoint_path = Path(CHECKPOINT_DIR) / f'{MODEL_NAME}_seed{run_seed}_best.pt'
    oom_occurred = False
    print(f'\n========== Seed {run_seed} ==========')

    for batch_size in BATCH_SIZE_CANDIDATES:
        gc.collect()
        torch.cuda.empty_cache()
        set_seed(run_seed)
        train_loader, val_loader, test_loader = build_experiment_loaders(batch_size)

        set_seed(run_seed)
        model = ResNet50Classifier(
            num_classes=NUM_CLASSES, pretrained=True, freeze_features=True
        ).to(DEVICE)

        print(f'Attempting seed {run_seed} with batch size {batch_size}...')
        try:
            history, best_epoch = fit_resnet50(
                model, loss_fn, train_loader, val_loader, checkpoint_path
            )
            actual_batch_size = batch_size
            break
        except RuntimeError as error:
            if 'out of memory' not in str(error).lower():
                raise
            oom_occurred = True
            print(f'CUDA OOM at batch size {batch_size}; retrying with a smaller batch size.')
            del model
            gc.collect()
            torch.cuda.empty_cache()
    else:
        raise RuntimeError(f'CUDA OOM at all batch sizes {BATCH_SIZE_CANDIDATES}.')

    assert checkpoint_path.exists(), f'Checkpoint not found: {checkpoint_path}'
    runs.append({
        'seed': run_seed, 'history': history, 'best_epoch': best_epoch,
        'batch_size': actual_batch_size, 'oom_occurred': oom_occurred,
        'checkpoint_path': checkpoint_path,
    })
    print(f'Best epoch: {best_epoch} | Batch size: {actual_batch_size} | OOM: {oom_occurred}')
    del model
    gc.collect()
    torch.cuda.empty_cache()

## 6. Training curves

In [ ]:
for run in runs:
    fig = plot_training_curves(
        run['history'], model_name=f"{MODEL_NAME}_seed{run['seed']}"
    )
    fig.show()

## 7. Reload the BEST checkpoint and evaluate on the test set

In [ ]:
seed_metrics = []
combined_confusion_matrix = [[0] * NUM_CLASSES for _ in range(NUM_CLASSES)]

for run in runs:
    model = ResNet50Classifier(
        num_classes=NUM_CLASSES, pretrained=False, freeze_features=False
    ).to(DEVICE)
    model.load_state_dict(torch.load(
        run['checkpoint_path'], map_location=DEVICE, weights_only=True
    ))
    test_loss, test_preds, test_labels = evaluate(model, test_loader, loss_fn)
    metrics = compute_metrics(test_labels, test_preds)
    metrics['loss'] = float(test_loss)
    metrics['seed'] = run['seed']
    seed_metrics.append(metrics)

    for row in range(NUM_CLASSES):
        for column in range(NUM_CLASSES):
            combined_confusion_matrix[row][column] += metrics['confusion_matrix'][row][column]

    print(f"Seed {run['seed']} | loss {test_loss:.4f} | "
          f"accuracy {metrics['accuracy']:.4f} | macro-F1 {metrics['f1_macro']:.4f}")
    save_metrics(metrics, model_name=f"{MODEL_NAME}_seed{run['seed']}")
    del model

summary = {'seeds': list(SEEDS)}
for metric_name in ('accuracy', 'f1_macro', 'loss'):
    values = [metrics[metric_name] for metrics in seed_metrics]
    summary[f'{metric_name}_mean'] = mean(values)
    summary[f'{metric_name}_std'] = stdev(values)

print('\nThree-seed test summary')
print(f"Accuracy : {summary['accuracy_mean']:.4f} ± {summary['accuracy_std']:.4f}")
print(f"Macro-F1 : {summary['f1_macro_mean']:.4f} ± {summary['f1_macro_std']:.4f}")
print(f"Loss     : {summary['loss_mean']:.4f} ± {summary['loss_std']:.4f}")
save_metrics(summary, model_name=f'{MODEL_NAME}_3seeds_summary')

## 8. Confusion matrix

In [ ]:
def format_mean_std(values):
    return f'{mean(values):.4f} ± {stdev(values):.4f}'

supports = {name: test_labels.count(index) for index, name in enumerate(CLASS_NAMES)}

print('Three-seed aggregate classification report')
print(f"Seeds: {', '.join(map(str, SEEDS))}")
print(f"Accuracy : {format_mean_std([item['accuracy'] for item in seed_metrics])}")
print(f"Macro-F1 : {format_mean_std([item['f1_macro'] for item in seed_metrics])}\n")
print(f"{'class':<14}{'precision':>20}{'recall':>20}{'f1-score':>20}{'support':>10}")

for class_name in CLASS_NAMES:
    precision = format_mean_std([item['per_class'][class_name]['precision'] for item in seed_metrics])
    recall = format_mean_std([item['per_class'][class_name]['recall'] for item in seed_metrics])
    f1 = format_mean_std([item['per_class'][class_name]['f1'] for item in seed_metrics])
    print(f'{class_name:<14}{precision:>20}{recall:>20}{f1:>20}{supports[class_name]:>10}')

for average_name, weighted in [('macro avg', False), ('weighted avg', True)]:
    row = {}
    for metric_name in ('precision', 'recall', 'f1'):
        values = []
        for item in seed_metrics:
            class_values = [item['per_class'][name][metric_name] for name in CLASS_NAMES]
            if weighted:
                values.append(sum(value * supports[name] for value, name in zip(class_values, CLASS_NAMES)) / len(test_labels))
            else:
                values.append(mean(class_values))
        row[metric_name] = format_mean_std(values)
    print(f"{average_name:<14}{row['precision']:>20}{row['recall']:>20}{row['f1']:>20}{len(test_labels):>10}")

fig_cm = plot_confusion_matrix(
    combined_confusion_matrix, model_name=f'{MODEL_NAME}_3seeds'
)
fig_cm.show()